In [1]:
from __future__ import annotations

import os
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd


In [2]:
FEATURE_NAMES = ("R1", "R2", "R3", "W1", "W2", "W3", "D")


def make_feature_importance(
    active_features: list[str] | None = None, default: float = 0.0
) -> dict[str, float]:
    """Create a feature-weight dictionary for the mock generator."""
    weights = {name: default for name in FEATURE_NAMES}
    if active_features is None:
        return {name: 1.0 for name in FEATURE_NAMES}

    for name in active_features:
        if name not in weights:
            raise ValueError(f"Unknown feature: {name}")
        weights[name] = 1.0
    return weights


def _sigmoid(x: float) -> float:
    return 1.0 / (1.0 + np.exp(-x))


def mock_welding_experiment(
    R1: float,
    R2: float,
    R3: float,
    W1: float,
    W2: float,
    W3: float,
    D: float,
    feature_importance: dict[str, float] | None = None,
    n_measurements: int = 1,
    stress_noise_std: float = 3.0,
    quality_noise_std: float = 0.35,
    seed: int | None = None,
) -> dict[str, Any]:
    """Mock welding response with controllable feature importance and noise.

    The same latent score drives both stress outputs and the quality label.
    Set a feature weight to 0.0 to remove its effect. For example, to make
    the outputs depend only on R1, pass feature_importance={"R1": 1.0}.
    """

    if n_measurements < 1:
        raise ValueError("n_measurements must be >= 1")

    rng = np.random.default_rng(seed)
    values = {
        "R1": float(R1),
        "R2": float(R2),
        "R3": float(R3),
        "W1": float(W1),
        "W2": float(W2),
        "W3": float(W3),
        "D": float(D),
    }

    weights = (
        make_feature_importance(list(FEATURE_NAMES))
        if feature_importance is None
        else {name: float(feature_importance.get(name, 0.0)) for name in FEATURE_NAMES}
    )

    # Normalize inputs to keep the output scale reasonably stable.
    normalized = {
        "R1": values["R1"] / 1000.0,
        "R2": values["R2"] / 1000.0,
        "R3": values["R3"] / 1000.0,
        "W1": values["W1"] / 100.0,
        "W2": values["W2"] / 100.0,
        "W3": values["W3"] / 100.0,
        "D": values["D"] / 10.0,
    }

    latent = sum(weights[name] * normalized[name] for name in FEATURE_NAMES)

    stress_advancing_mean = (
        140.0
        + 42.0 * latent
        + 7.0 * np.sin(2.5 * latent)
    )
    stress_retreating_mean = (
        128.0
        + 36.0 * latent
        + 6.0 * np.cos(2.0 * latent)
    )

    stress_advancing = stress_advancing_mean + rng.normal(
        0.0, stress_noise_std, size=n_measurements
    )
    stress_retreating = stress_retreating_mean + rng.normal(
        0.0, stress_noise_std, size=n_measurements
    )

    quality_score = 1.6 - abs(latent - 1.25) + rng.normal(0.0, quality_noise_std)
    visual_quality = bool(rng.random() < _sigmoid(quality_score))

    return {
        "stress_advancing_side": [float(x) for x in stress_advancing],
        "stress_retreating_side": [float(x) for x in stress_retreating],
        "visual_quality": visual_quality,
    }


example = mock_welding_experiment(
    R1=1200,
    R2=1350,
    R3=1450,
    W1=15,
    W2=18,
    W3=16,
    D=6,
    feature_importance={"R1": 1.0},
    n_measurements=3,
    seed=3423,
)
example

{'stress_advancing_side': [191.37259687599598,
  191.5458106552477,
  187.24070048424335],
 'stress_retreating_side': [164.1466547533231,
  162.1347536344478,
  163.55366923325775],
 'visual_quality': True}

In [3]:
folder_path = Path(os.getcwd()) / "data"

In [4]:
exp = pd.read_excel(folder_path / "experiments.xlsx")
display(exp)

,trial_index,R1,R2,R3,W1,W2,W3,D,visual_quality,sigma_advancing_1,sigma_advancing_2,sigma_advancing_3,sigma_retreating_1,sigma_retreating_2,sigma_retreating_3
0,0,1700,1700,1700,70,70,70,20,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,1462,1811,1851,93,63,41,17,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,1821,1598,1421,62,74,99,20,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,1894,1873,1756,73,98,82,14,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,1691,1519,1622,51,41,57,29,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,5,1561,1943,1492,42,48,47,12,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,6,1957,1466,1959,83,90,92,26,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,7,1730,1704,1664,64,83,75,19,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,8,1489,1687,1835,92,57,66,23,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,9,1549,1920,1641,78,74,59,11,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
print(exp.columns)
cols_advancing = [
    "sigma_advancing_1",
    "sigma_advancing_2",
    "sigma_advancing_3",
]
cols_retreating = [
    "sigma_retreating_1",
    "sigma_retreating_2",
    "sigma_retreating_3",
]

Index(['trial_index', 'R1', 'R2', 'R3', 'W1', 'W2', 'W3', 'D',
       'visual_quality', 'sigma_advancing_1', 'sigma_advancing_2',
       'sigma_advancing_3', 'sigma_retreating_1', 'sigma_retreating_2',
       'sigma_retreating_3'],
      dtype='str')


In [6]:
filt_advancing = exp[cols_advancing].isna().all(axis=1)
filt_retreating = exp[cols_retreating].isna().all(axis=1)
filt_removed = filt_advancing | filt_retreating
filt_removed

0    True
1    True
2    True
3    True
4    True
5    True
6    True
7    True
8    True
9    True
dtype: bool

In [7]:
exp_todo = exp[filt_removed].copy()
exp
display(exp)
display(exp_todo)

,trial_index,R1,R2,R3,W1,W2,W3,D,visual_quality,sigma_advancing_1,sigma_advancing_2,sigma_advancing_3,sigma_retreating_1,sigma_retreating_2,sigma_retreating_3
0,0,1700,1700,1700,70,70,70,20,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,1462,1811,1851,93,63,41,17,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,1821,1598,1421,62,74,99,20,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,1894,1873,1756,73,98,82,14,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,1691,1519,1622,51,41,57,29,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,5,1561,1943,1492,42,48,47,12,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,6,1957,1466,1959,83,90,92,26,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,7,1730,1704,1664,64,83,75,19,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,8,1489,1687,1835,92,57,66,23,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,9,1549,1920,1641,78,74,59,11,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,trial_index,R1,R2,R3,W1,W2,W3,D,visual_quality,sigma_advancing_1,sigma_advancing_2,sigma_advancing_3,sigma_retreating_1,sigma_retreating_2,sigma_retreating_3
0,0,1700,1700,1700,70,70,70,20,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,1462,1811,1851,93,63,41,17,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,1821,1598,1421,62,74,99,20,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,1894,1873,1756,73,98,82,14,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,1691,1519,1622,51,41,57,29,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,5,1561,1943,1492,42,48,47,12,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,6,1957,1466,1959,83,90,92,26,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,7,1730,1704,1664,64,83,75,19,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,8,1489,1687,1835,92,57,66,23,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,9,1549,1920,1641,78,74,59,11,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
for idx, row in exp_todo.iterrows():
    result = mock_welding_experiment(
        R1=row["R1"],
        R2=row["R2"],
        R3=row["R3"],
        W1=row["W1"],
        W2=row["W2"],
        W3=row["W3"],
        D=row["D"],
        feature_importance={"R1": 1.0},
        n_measurements=len(cols_advancing),
        seed=idx,
    )
    exp.loc[idx, cols_advancing] = result["stress_advancing_side"]
    exp.loc[idx, cols_retreating] = result["stress_retreating_side"]
    exp.loc[idx, "visual_quality"] = float(result["visual_quality"])


In [9]:
exp.to_excel(folder_path / "experiments.xlsx", index=False)